# NBA Fantasy Points Predictor

Predicting DraftKings fantasy points per player per game using recent game history.

DK scoring: PTS + 1.25*REB + 1.5*AST + 2*STL + 2*BLK - 0.5*TOV + 0.5*FG3M

In [17]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, train_test_split, TimeSeriesSplit
from sklearn.metrics import mean_squared_error

from pathlib import Path

DATA_DIR = Path("data")

## Data Loading

In [18]:
player_logs = pd.read_csv(DATA_DIR / "nba_player_game_logs.csv")
games       = pd.read_csv(DATA_DIR / "nba_historical_games.csv")

# Parse dates
player_logs["GAME_DATE"] = pd.to_datetime(player_logs["GAME_DATE"])
games["date"]            = pd.to_datetime(games["date"])

# Normalize GAME_ID to int for joining (player logs have leading zeros: '0029900001')
player_logs["game_id_int"] = player_logs["GAME_ID"].astype(int)
games["game_id_int"]       = games["game_id"].astype(int)

# Convert MIN to numeric (NBA API occasionally returns "30:45" string format)
player_logs["MIN"] = pd.to_numeric(player_logs["MIN"], errors="coerce")

print("Player logs:", player_logs.shape)
print("Games:      ", games.shape)
player_logs.head(3)

Player logs: (671862, 25)
Games:       (32357, 47)


,SEASON_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,GAME_ID,GAME_DATE,MIN,PTS,REB,...,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,PLUS_MINUS,game_id_int
0,21999,431,Shawn Kemp,1610612739,CLE,29900001,1999-11-02,31,17,5,...,20,0.300,0,0,NaN,5,8,0.625,2,29900001
1,21999,95,Mark Bryant,1610612739,CLE,29900001,1999-11-02,15,3,3,...,3,0.333,0,0,NaN,1,2,0.500,-24,29900001
2,21999,1538,Cedric Henderson,1610612739,CLE,29900001,1999-11-02,20,0,2,...,1,0.000,0,0,NaN,0,0,NaN,-17,29900001


## Target

DK scoring: PTS x 1.0, REB x 1.25, AST x 1.5, STL x 2.0, BLK x 2.0, TOV x -0.5, FG3M x 0.5

In [19]:
player_logs["FANTASY_PTS"] = (
    player_logs["PTS"]  * 1.00 +
    player_logs["REB"]  * 1.25 +
    player_logs["AST"]  * 1.50 +
    player_logs["STL"]  * 2.00 +
    player_logs["BLK"]  * 2.00 +
    player_logs["TOV"]  * -0.50 +
    player_logs["FG3M"] * 0.50
)

print(f"Fantasy points - mean: {player_logs['FANTASY_PTS'].mean():.2f}, "
      f"std: {player_logs['FANTASY_PTS'].std():.2f}, "
      f"min: {player_logs['FANTASY_PTS'].min():.2f}, "
      f"max: {player_logs['FANTASY_PTS'].max():.2f}")

Fantasy points - mean: 20.54, std: 14.15, min: -2.00, max: 107.75


## Feature Engineering

Using the last 10 games as features for each player, their team, and the opponent.

### Player stats (lags 1-10)

In [20]:
PLAYER_STATS = ["PTS", "REB", "AST", "STL", "BLK", "TOV", "FG3M", "MIN"]

# sort first or shift() won't look back in time correctly
player_logs = player_logs.sort_values(["PLAYER_ID", "GAME_DATE"]).reset_index(drop=True)

player_lag_cols = []
for stat in PLAYER_STATS:
    for lag in range(1, 11):
        col = f"player_{stat}_lag{lag}"
        player_logs[col] = player_logs.groupby("PLAYER_ID")[stat].shift(lag)
        player_lag_cols.append(col)

print(f"player lag features: {len(player_lag_cols)}")
player_logs[["PLAYER_NAME", "GAME_DATE"] + player_lag_cols[:4]].head()

player lag features: 80


,PLAYER_NAME,GAME_DATE,player_PTS_lag1,player_PTS_lag2,player_PTS_lag3,player_PTS_lag4
0,Grant Long,1999-12-27,NaN,NaN,NaN,NaN
1,Grant Long,1999-12-29,4.0,NaN,NaN,NaN
2,Grant Long,1999-12-30,4.0,4.0,NaN,NaN
3,Grant Long,2000-01-04,4.0,4.0,4.0,NaN
4,Grant Long,2000-01-05,1.0,4.0,4.0,4.0


### Team stats (lags 1-10)

The games table has separate home/away columns so I stack them into one row per team first.

In [21]:
TEAM_STATS = ["score", "fg_made", "fg3_made", "reb", "ast", "stl", "blk", "tov"]

home_games = games[["game_id_int", "date", "home_team",
                     "home_score", "home_fg_made", "home_fg3_made",
                     "home_reb", "home_ast", "home_stl",
                     "home_blk", "home_tov"]].copy()
home_games.columns = ["game_id_int", "date", "team"] + TEAM_STATS

away_games = games[["game_id_int", "date", "away_team",
                     "away_score", "away_fg_made", "away_fg3_made",
                     "away_reb", "away_ast", "away_stl",
                     "away_blk", "away_tov"]].copy()
away_games.columns = ["game_id_int", "date", "team"] + TEAM_STATS

team_games = pd.concat([home_games, away_games], ignore_index=True)
team_games = team_games.sort_values(["team", "date"]).reset_index(drop=True)

print(f"Team-game rows (should be 2 x {len(games):,}): {len(team_games):,}")
team_games.head(3)

Team-game rows (should be 2 x 32,357): 64,714


,game_id_int,date,team,score,fg_made,fg3_made,reb,ast,stl,blk,tov
0,29900003,1999-11-02,ATL,87,31,2,50,15,5,5,23
1,29900021,1999-11-04,ATL,109,41,6,46,22,3,5,26
2,29900037,1999-11-06,ATL,113,44,3,42,21,10,3,12


In [22]:
team_lag_cols = []
for stat in TEAM_STATS:
    for lag in range(1, 11):
        col = f"team_{stat}_lag{lag}"
        team_games[col] = team_games.groupby("team")[stat].shift(lag)
        team_lag_cols.append(col)

print(f"team lag features: {len(team_lag_cols)}")

team lag features: 80


### Opponent stats (lags 1-10)

In [23]:
# figure out who each team played in each game
opp_map = pd.concat([
    games[["game_id_int", "home_team", "away_team"]].rename(
        columns={"home_team": "team", "away_team": "opponent"}),
    games[["game_id_int", "away_team", "home_team"]].rename(
        columns={"away_team": "team", "home_team": "opponent"})
], ignore_index=True)

team_games = team_games.merge(opp_map, on=["game_id_int", "team"], how="left")
team_games[["team", "opponent", "date"]].head(3)

,team,opponent,date
0,ATL,WAS,1999-11-02
1,ATL,MIL,1999-11-04
2,ATL,CHI,1999-11-06


In [24]:
# look up each opponent's lag features for the same game
opp_lag_cols = [f"opp_{s}_lag{l}" for s in TEAM_STATS for l in range(1, 11)]

opp_lookup = team_games[["game_id_int", "team"] + team_lag_cols].rename(
    columns={"team": "opponent"}
)
opp_lookup.columns = ["game_id_int", "opponent"] + opp_lag_cols

team_games = team_games.merge(opp_lookup, on=["game_id_int", "opponent"], how="left")
print(f"opp lag features: {len(opp_lag_cols)}")

opp lag features: 80


### Build feature matrix

Merge player + team + opponent lags. Drop rows missing lag history and short appearances (< 10 min).

In [25]:
team_cols = ["game_id_int", "team"] + team_lag_cols + opp_lag_cols

df = player_logs.merge(
    team_games[team_cols],
    left_on=["game_id_int", "TEAM_ABBREVIATION"],
    right_on=["game_id_int", "team"],
    how="inner"
)

all_feature_cols = player_lag_cols + team_lag_cols + opp_lag_cols

df_clean = df.dropna(subset=all_feature_cols).reset_index(drop=True)
df_clean = df_clean[df_clean["MIN"] >= 10].reset_index(drop=True)

print(f"{len(df_clean):,} rows, {len(all_feature_cols)} features")

X = df_clean[all_feature_cols]
y = df_clean["FANTASY_PTS"]

561,108 rows, 240 features


In [26]:
# Quick look at the target
print(f"Fantasy points - mean: {y.mean():.2f}, std: {y.std():.2f}, "
      f"min: {y.min():.2f}, max: {y.max():.2f}")

# Lag-1 feature correlations with target
lag1_cols = [f"player_{s}_lag1" for s in PLAYER_STATS]
corrs = df_clean[lag1_cols + ["FANTASY_PTS"]].corr()["FANTASY_PTS"].drop("FANTASY_PTS")
print("Lag-1 correlations with FANTASY_PTS:")
print(corrs.sort_values(ascending=False).to_string())

Fantasy points - mean: 23.57, std: 13.22, min: -2.00, max: 107.75
Lag-1 correlations with FANTASY_PTS:
player_PTS_lag1     0.515653
player_MIN_lag1     0.502866
player_AST_lag1     0.376435
player_TOV_lag1     0.348241
player_REB_lag1     0.344978
player_STL_lag1     0.192338
player_FG3M_lag1    0.177160
player_BLK_lag1     0.147097


## Train/Test Split

Splitting by date so we're not training on future games. Everything before 2023-24 is train, rest is test.

In [27]:
train_mask = df_clean["GAME_DATE"] < "2023-10-01"

X_train = X[train_mask]
X_test  = X[~train_mask]
y_train = y[train_mask]
y_test  = y[~train_mask]

print(f"Train: {len(X_train):,} rows, Test: {len(X_test):,} rows")

Train: 496,750 rows, Test: 64,358 rows


## Linear Regression Baseline

Using TimeSeriesSplit for CV so future folds don't leak into training.

In [28]:
# 1. choose model class (LinearRegression, already imported above)
# 2. instantiate model
model = make_pipeline(StandardScaler(), LinearRegression())

# 3. fit model to data
model.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('linearregression', LinearRegression())])

In [29]:
# 4. predict on training/testing data, evaluate
train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
test_rmse  = np.sqrt(mean_squared_error(y_test,  model.predict(X_test)))

print(f"Train RMSE: {train_rmse:.3f} fantasy points")
print(f"Test  RMSE: {test_rmse:.3f} fantasy points")

Train RMSE: 9.427 fantasy points
Test  RMSE: 9.862 fantasy points


In [30]:
tscv = TimeSeriesSplit(n_splits=5)

cv_rmses = np.sqrt(
    -cross_val_score(model, X_train, y_train, cv=tscv,
                     scoring="neg_mean_squared_error")
)

print(f"CV RMSE (TimeSeriesSplit): {cv_rmses.mean():.3f} +/- {cv_rmses.std():.3f}")
print(f"per fold: {np.round(cv_rmses, 3)}")

CV RMSE (TimeSeriesSplit): 9.465 +/- 0.104
per fold: [9.397 9.393 9.43  9.434 9.669]


In [31]:
print(f"train RMSE: {train_rmse:.3f}  test RMSE: {test_rmse:.3f}  CV: {cv_rmses.mean():.3f} +/- {cv_rmses.std():.3f}")
print(f"naive baseline: {y_test.std():.3f}  improvement: {(1 - test_rmse / y_test.std()) * 100:.1f}%")

train RMSE: 9.427  test RMSE: 9.862  CV: 9.465 +/- 0.104
naive baseline: 13.877  improvement: 28.9%
